# RSA on GPU (group steps 3, 5, 6, 7, 8) — v3.1.0

> **Check the version.** Cell 4 prints the version of all three files and
> **stops** if they disagree. Colab caches imported modules, so after copying a
> new `gpu_group.py` to Drive you must **Runtime → Restart session** before the
> new one is picked up — a version mismatch here almost always means a stale
> module in a live runtime rather than a stale file on Drive.

Takes the **per-participant** maps a previous Colab run already wrote to Drive
(`result_<model>_<specie>-sub-NN.zip`, from steps 1/2/4) and reduces them all the
way to what pipeline steps 9-10 need.

**No package needed.** The result zips are read **in place** on Drive — never
unpacked, never rebuilt. Every pipeline parameter (dataset, GLM model, RSA model,
specie, radius, `dis_method`, `rsa_method`, `mah_fold`, the per-run layout, which
permutation indices each participant has) is recovered from the arcnames *inside*
those zips. The only things that cannot be — the searchlight mask and the config's
participant list — come from a small committed folder, `tools/colab_gpu/refs/`
(~80 kB), so a run never touches the `P:` share.

| Step | What it produces | Leaves Colab? |
|---|---|---|
| 3 | group mean/std of the real model-similarity maps | yes, 2 maps |
| 5 | `reps_group` group permutation maps | **no** — inputs to 6/7 only |
| 6 | the voxelwise null distribution (mean/std) | yes, 2 maps |
| 7 | a z map per permutation, plus the real z map | only the **real** one |
| 8 | cluster-size distribution at several z thresholds | yes, one small `.npy` |

## Why the bulky maps stay here

Measured on EmoC humans at `reps_group=1000`, per model:

| Output | Size | Read by |
|---|---|---|
| step 5 group means (1000) | 727 MB | steps 6-7 — both run here |
| step 7 rnd z maps (1000) | 1273 MB | step 8 — runs here |
| step 3 + 6 + real z + step 8 `.npy` | **~4 MB** | steps 9-10, on the workstation |

So a model ships ~4 MB instead of ~2 GB. Over a ~90-model battery that is roughly
**0.4 GB instead of 180 GB** crossing Drive, and no multi-gigabyte upload per
model — which is most of the wall-clock time.

**Runtime:** GPU (L4 or T4). Pick **High-RAM** for humans — the run holds
`reps_group x n_mask_voxels` float64 arrays.

**Setup (once):** copy the code and the refs folder to Drive, from the workstation
(Anaconda Prompt):

```
copy \github\dog_brain_toolkit\tools\colab_gpu\gpu_rsa.py "G:\My Drive\rsa_colab\"
copy \github\dog_brain_toolkit\tools\colab_gpu\gpu_group.py "G:\My Drive\rsa_colab\"
copy \github\dog_brain_toolkit\tools\colab_gpu\run_colab_group.py "G:\My Drive\rsa_colab\"
xcopy /E /I /Y \github\dog_brain_toolkit\tools\colab_gpu\refs "G:\My Drive\rsa_colab\refs"
```

In [ ]:
# 1. Check the GPU and install nibabel (torch and scipy are preinstalled on Colab).
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!pip -q install nibabel
import scipy.ndimage  # step 8 needs it
print('scipy OK')

In [ ]:
# 2. Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. EDIT THESE.
CODE_DIR    = '/content/drive/MyDrive/rsa_colab'            # gpu_group.py, run_colab_group.py, gpu_rsa.py
REFS_DIR    = '/content/drive/MyDrive/rsa_colab/refs'       # the committed refs folder
RESULTS_DIR = '/content/drive/MyDrive/rsa_colab/results'    # result_<model>_<specie>-sub-NN.zip, read in place
OUT_DIR     = '/content/drive/MyDrive/rsa_colab/results'    # result_group_*.zip go here (can be the same)

SPECIE      = 'H'      # required: the results folder holds one species per run
MODELS      = None     # None = every model found in RESULTS_DIR for this specie
REPS_GROUP  = 1000     # group permutations (searchlight.py --reps_group)

# The availability gate. The denominator is the config's participant list from
# REFS_DIR (EmoC: 40 for H, 15 for D), so this is the same check the CPU makes.
# As of 2026-09-05 all 91 EmoC H models are at 40/40, so 1.0 is fine; 0.8 leaves
# room for a model that is still finishing. Cell 5 shows which models pass.
MIN_PERCENTAGE_AVAILABLE = 0.8

STEPS       = [3, 5, 6, 7, 8]  # 7's real z map needs 3; 8 needs 7 in the same run
BATCH       = 20000            # voxels per GPU chunk; lower if you hit out-of-memory
G_BATCH     = 64               # group permutations gathered per pass
WORKERS     = 8                # threads for reading zips / writing niftis / labelling

# Step 8 cluster-forming thresholds, all computed in ONE pass and stored under
# separate keys, so step 9 can pick any of them later with --z_threshold.
# LIST EVERY ONE YOU MIGHT WANT: the z maps are not kept, so adding a threshold
# afterwards means re-running the model.
Z_THRESHOLDS = [3.1, 3.5, 4.0, 4.5, 5.0]

# Copy each model's result zips to local disk before reading their members.
# MEASURED A NET LOSS on a Windows Drive mount and off by default: two cold
# models, 38 zips / ~1.6 GB each, 8 threads --
#     direct   128.0s (12.6 MB/s)
#     prefetch 175.9s (copy 119.8s + local re-read 56.1s)
# The mount turned out to be bandwidth-bound, not latency-bound, so direct
# member reads already run at copy speed and prefetch just reads the same
# bytes twice. Worth retrying on Colab (different FUSE layer) -- time one
# model each way before committing to it for a whole battery.
PREFETCH_ZIPS = False

# The two bulky intermediates -- see the note at the top. Turn WRITE_Z_MAPS on
# only if you intend to run step 8 on the workstation instead.
WRITE_GROUP_MEANS = False   # step 5's 1000 maps (~727 MB/model, humans)
WRITE_Z_MAPS      = False   # step 7's 1000 maps (~1273 MB/model, humans)

In [ ]:
# 4. Load the code from Drive and CHECK THE VERSIONS.
#    gpu_group.py, run_colab_group.py and this notebook are copied to Drive
#    separately, so any one of them can be stale. This stops the run if they
#    disagree, and names the file to re-copy.
NOTEBOOK_VERSION = "3.1.0"

import sys, os, importlib
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import gpu_group, run_colab_group
# picks up a newer file if you re-copied one without restarting; a cached module
# that is already imported still needs Runtime -> Restart session
importlib.reload(gpu_group)
importlib.reload(run_colab_group)

gpu_group.check_versions(NOTEBOOK_VERSION, strict=True)
print("
versions agree — safe to run.")

In [ ]:
# 5. Preflight: probe every model and show the gate verdict.
#
#    ONE zip is opened per model (91 for the EmoC human battery). That is ~2 s
#    locally but ~175 s on Colab's Drive, so the result is kept in PER_MODEL and
#    handed to cell 6 rather than probed a second time.
#
#    Each model gets its OWN manifest. This folder holds two analyses -- the
#    mahalanobis battery is per-participant, the correlation one is per-run -- and
#    applying one model's parameters to the other half silently builds paths that
#    do not exist.
PER_MODEL = gpu_group.discover_model_manifests(
    RESULTS_DIR, REFS_DIR, specie=SPECIE, models=MODELS,
    reps_group=REPS_GROUP, min_percentage_available=MIN_PERCENTAGE_AVAILABLE)

mask_img, mask_bool = gpu_group.load_group_mask(
    next(iter(PER_MODEL.values())), refs_dir=REFS_DIR)
print(f"\nmask: {mask_bool.shape}, {int(mask_bool.sum())} voxels")

groups = {}
for name, m in PER_MODEL.items():
    groups.setdefault(gpu_group.manifest_signature(m), []).append(name)

n_run = 0
for sig, names in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    dis, fold, rsa, rad, per_run = sig[2], sig[3], sig[4], sig[5], sig[7]
    m0 = PER_MODEL[names[0]]
    n_units = len(gpu_group.units(m0))
    n_maps = n_units * m0["reps"]
    gb = n_maps * int(mask_bool.sum()) * 8 / 1e9
    print(f"\n=== {len(names)} model(s): {dis}"
          + (f"/{fold}" if dis == "mahalanobis" else "")
          + f"/{rsa}  r-{rad}  " + ("per-run" if per_run else "per-participant")
          + f"  {n_units} unit(s), reps={m0['reps']} ===")
    print(f"    ~{n_maps} map(s) per model -> {gb:.1f} GB held in RAM")
    if gb > 25:
        print(f"    !! WARNING: {gb:.0f} GB will not fit a Colab High-RAM runtime "
              f"(~51 GB, and loading transiently needs double). Expect an OOM.")
    for name in sorted(names):
        m = PER_MODEL[name]
        have = len(m["zips_per_model"][name])
        total = len(m["participants"])
        pct = 100 * have / total
        ok = pct >= MIN_PERCENTAGE_AVAILABLE * 100
        n_run += ok
        print(f"    [{'run ' if ok else 'SKIP'}] {name:45s} {have:3d}/{total} "
              f"participant(s) = {pct:5.1f}%")

print(f"\n{n_run} model(s) pass the gate, {len(PER_MODEL) - n_run} will be skipped.")
print(f"estimated download: ~{n_run * 4} MB total")

In [ ]:
# 6. Run. Resumable: skips models whose result_group_*.zip already exists, so
#    just re-run this cell after a disconnect.
#
#    manifests=PER_MODEL reuses cell 5's probe instead of repeating it (~175 s on
#    Colab). Pass models=<list> to control the ORDER -- a second Colab instance can
#    run `list(reversed(sorted(PER_MODEL)))` to work from the far end of the list.
RUN_ORDER = sorted(PER_MODEL)          # reversed(sorted(PER_MODEL)) for instance 2

written = run_colab_group.run_group_package(
    None, RESULTS_DIR, OUT_DIR,
    refs_dir=REFS_DIR, specie=SPECIE, models=RUN_ORDER, manifests=PER_MODEL,
    reps_group=REPS_GROUP,
    min_percentage_available=MIN_PERCENTAGE_AVAILABLE,
    work_root='/content/group_work',
    steps=STEPS, batch=BATCH, g_batch=G_BATCH, workers=WORKERS,
    write_group_means=WRITE_GROUP_MEANS, write_z_maps=WRITE_Z_MAPS,
    z_thresholds=Z_THRESHOLDS, prefetch_zips=PREFETCH_ZIPS, verbose=True)
print('\nnew group result zips:')
for w in written:
    print(' ', w)

## Back on the workstation

```
python \github\dog_brain_toolkit\tools\unpack_results.py \path\to\downloads --dry-run
python \github\dog_brain_toolkit\tools\unpack_results.py \path\to\downloads
python \github\dog_brain_toolkit\searchlight.py --dataset EmoC --model basic-block --specie H --rsa_model action_tendency__all --steps_to_run 9 10 --z_threshold 4.0
```

Step 8 is already done, so go straight to 9 and 10, with `--z_threshold` set to any
value in `Z_THRESHOLDS`.

**One thing to watch:** `unpack_results.py` skips files that already exist unless you
pass `--replace`. If a model already has a `dist/..._dist.npy` on the data disk from
an earlier workstation run, the one computed here will not land. Either delete that
file first or unpack with `--replace` — the Colab `.npy` carries every threshold in
`Z_THRESHOLDS`, so nothing is lost by replacing it unless the old file held a
threshold you did not list.

**Keeping the refs current.** `refs/` only changes when a dataset's config
participant list does. Rebuild it on the workstation while the share is up, then
re-copy to Drive:

```
python \github\dog_brain_toolkit\tools\colab_gpu\refs\build_refs.py
```